In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Silver Layer Script

### Data Access Using App

In [ ]:

spark.conf.set("fs.azure.account.auth.type.projectstorage8896.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.projectstorage8896.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.projectstorage8896.dfs.core.windows.net","<YOUR-CLIENT-ID>")
spark.conf.set("fs.azure.account.oauth2.client.secret.projectstorage8896.dfs.core.windows.net","<YOUR-CLIENT-SECRET>"
)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.projectstorage8896.dfs.core.windows.net", "https://login.microsoftonline.com/<YOUR-TENANT-ID>/oauth2/token")

#### Read Calender Data

In [ ]:
df_cal = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@projectstorage8896.dfs.core.windows.net/AdventureWorks_Calendar")


In [ ]:
df_cust = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@projectstorage8896.dfs.core.windows.net/AdventureWorks_Customers")


In [ ]:
df_procat = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@projectstorage8896.dfs.core.windows.net/AdventureWorks_Product_Categories")


In [ ]:
df_prosub = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@projectstorage8896.dfs.core.windows.net/AdventureWorks_Product_Subcategories")

In [ ]:
df_prod = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@projectstorage8896.dfs.core.windows.net/AdventureWorks_Products")

In [ ]:
df_sales = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@projectstorage8896.dfs.core.windows.net/AdventureWorks_Sales*")

In [ ]:
df_ter = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@projectstorage8896.dfs.core.windows.net/AdventureWorks_Territories")

In [ ]:
df_ret = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@projectstorage8896.dfs.core.windows.net/Returns")

#### transformation

In [ ]:
df_cal.display()

In [ ]:
df_cal = df_cal.withColumn('Month',month(col('Date')))\
    .withColumn('Year',year(col('Date')))
df_cal.display()

In [ ]:
df_cal.write.format('parquet')\
    .mode('append')\
    .option('path','abfss://silver@projectstorage8896.dfs.core.windows.net/AdventureWorks_Calendar')\
    .save()

In [ ]:
df_cust.display()

In [ ]:
 from pyspark.sql.functions import concat, col, lit
 df_cust.withColumn("fullname",concat(col("Prefix"),lit(" "),col("FirstName"),lit(" "),col("LastName"))).display()



In [ ]:
df_cust.write.format('parquet')\
    .mode('append')\
    .option('path','abfss://silver@projectstorage8896.dfs.core.windows.net/AdventureWorks_Customers')\
    .save()

#### sub categories

In [ ]:
df_prosub.display()

In [ ]:
df_prosub.write.format('parquet')\
    .mode('append')\
    .option('path','abfss://silver@projectstorage8896.dfs.core.windows.net/AdventureWorks_Product_Subcategories')\
    .save()

#### product categories

In [ ]:
df_procat.write.format('parquet')\
    .mode('append')\
    .option('path','abfss://silver@projectstorage8896.dfs.core.windows.net/AdventureWorks_Product_Categories')\
    .save()

In [ ]:
df_prod.display()


In [ ]:
df_prod = df_prod.withColumn('ProductSKU',split(col('ProductSKU'),'-')[0])\
                .withColumn('ProductName',split(col('ProductName'),'-')[0])

In [ ]:
df_prod.display()

In [ ]:
df_prod.write.format('parquet')\
    .mode('append')\
    .option('path','abfss://silver@projectstorage8896.dfs.core.windows.net/AdventureWorks_Products')\
    .save()

In [ ]:
df_ret.display()

In [ ]:
df_ret.write.format('parquet')\
    .mode('append')\
    .option('path','abfss://silver@projectstorage8896.dfs.core.windows.net/Returns')\
    .save()

In [ ]:
df_ter.display()


In [ ]:
df_ter.write.format('parquet')\
    .mode('append')\
    .option('path','abfss://silver@projectstorage8896.dfs.core.windows.net/AdventureWorks_Territories')\
    .save()

In [ ]:
df_sales.display()


In [ ]:
df_sales = df_sales.withColumn('StockDate',to_timestamp('StockDate'))

In [ ]:
df_sales = df_sales.withColumn('OrderNumber',regexp_replace(col("OrderNumber"),'S','T'))

In [ ]:
df_sales = df_sales.withColumn("multiply",col('OrderLineItem')*col("OrderQuantity"))

In [ ]:
df_sales.display()

In [ ]:
df_sales.write.format('parquet')\
    .mode('append')\
    .option('path','abfss://silver@projectstorage8896.dfs.core.windows.net/AdventureWorks_Sales')\
    .save()

#### Sales Analysis

In [ ]:
df_sales.groupBy('OrderDate').agg(count('OrderNumber').alias('total_orders')).display()


####Prduct Category

In [ ]:
df_procat.display()

####Territories

In [ ]:
df_ter.display()